# Run the pyWOMBAT biogeochemical model in 1D water column configuration

In [1]:
import sys
import os
import itertools
import logging
from datetime import datetime
import numpy as np
import pandas as pd
import xarray as xr
import scipy as sci
from scipy.stats import qmc
import PyCO2SYS as pyco2
import matplotlib.pyplot as plt
import multiprocessing

# Ensure we are in the correct directory
#### NOTE! ### Change this to your own directory where you downloaded the code
os.chdir("/home/581/pjb581/py-WOMBAT/lite")
print(os.getcwd())
from main import main  # Import the main function from main.py

# Define output directory and ensure it exists (MAKE SURE A FORWARD SLASH EXISTS AT END)
### NOTE! ### Change the following to an output directory where you want the output saved
OUTPUT_DIR = "/g/data/es60/pjb581/py-WOMBAT/output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("python version =",sys.version[:5])
print("numpy version =", np.__version__)
print("xarray version =", xr.__version__)
print("scipy version =", sci.__version__)
print("PyCO2SYS version =", pyco2.__version__)

print(datetime.now())

"""
NOTE: Need read permissions for groups vk83, gb6, qv56

vk83 - gives access to initialisation files for NO3, DIC, Alk and Fe called in `initialisation.py`
gb6 - gives access to BRAN2020 files, specifically mixed-layer depth and w velocities in `mld_timeseries.py` and `w_timeseries.py`
qv56 - gives access to atmospheric forcing fields, specifically tas, uas, vas and rsds in `tas_timeseries.py`, `uas_timeseries.py`, `vas_timeseries.py` and `rsds_timeseries.py`

"""

/home/581/pjb581/py-WOMBAT/lite
python version = 3.10.
numpy version = 1.24.4
xarray version = 2023.8.0
scipy version = 1.15.1
PyCO2SYS version = 1.8.3.4
2025-07-18 09:58:27.335795


'\nNOTE: Need read permissions for groups vk83, gb6, qv56\n\nvk83 - gives access to initialisation files for NO3, DIC, Alk and Fe called in `initialisation.py`\ngb6 - gives access to BRAN2020 files, specifically mixed-layer depth and w velocities in `mld_timeseries.py` and `w_timeseries.py`\nqv56 - gives access to atmospheric forcing fields, specifically tas, uas, vas and rsds in `tas_timeseries.py`, `uas_timeseries.py`, `vas_timeseries.py` and `rsds_timeseries.py`\n\n'

### Check git branch

In [2]:
!git branch

  master
* pyWOMBAT-on-Gadi
  pyWOMBAT-on-Gadi_microbes


### Check that you can access the directories of vk83, gb6 and qv56 on Gadi

In [3]:
print("Fe initialisation")
os.listdir("/g/data/vk83/experiments/inputs/access-om3/wombat/initial_conditions/global.100km/2021.06.07")


Fe initialisation


['.manifest.yaml', 'FEMIP_model_median_iron_2016_fillmiss.nc']

In [8]:
print("NO3, DIC, Alkalinity initialisation")
contents = os.listdir("/g/data/vk83/experiments/inputs/access-om3/wombat/initial_conditions/global.100km/2024.04.02")
print(contents[0:2])

NO3, DIC, Alkalinity initialisation
['.manifest.yaml', 'GLODAPv2.2016b.PI_TCO2_fillmiss.nc']


In [7]:
print("uas (zonal wind speed)")
contents = os.listdir("/g/data/qv56/replicas/input4MIPs/CMIP6/OMIP/MRI/MRI-JRA55-do-1-5-0/atmos/3hrPt/uas/gr/v20200916/")
print(contents[0])

uas (zonal wind speed)
uas_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_201901010000-201912312100.nc


In [9]:
print("vas (zonal wind speed)")
contents = os.listdir("/g/data/qv56/replicas/input4MIPs/CMIP6/OMIP/MRI/MRI-JRA55-do-1-5-0/atmos/3hrPt/vas/gr/v20200916/")
print(contents[0:2])

vas (zonal wind speed)
['vas_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_200101010000-200112312100.nc', 'vas_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_196301010000-196312312100.nc']


In [10]:
print("tas (surface atmospheric temperature)")
contents = os.listdir("/g/data/qv56/replicas/input4MIPs/CMIP6/OMIP/MRI/MRI-JRA55-do-1-5-0/atmos/3hrPt/tas/gr/v20200916/")
print(contents[0:2])

tas (surface atmospheric temperature)
['tas_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_195801010000-195812312100.nc', 'tas_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_197801010000-197812312100.nc']


In [11]:
print("rsds (downwelling shortwave radiation)")
contents = os.listdir("/g/data/qv56/replicas/input4MIPs/CMIP6/OMIP/MRI/MRI-JRA55-do-1-5-0/atmos/3hr/rsds/gr/v20200916/")
print(contents[0:2])

rsds (downwelling shortwave radiation)
['rsds_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_195901010130-195912312230.nc', 'rsds_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_200801010130-200812312230.nc']


In [13]:
print("BRAN2020 w velocities and mixed layer depth")
contents = os.listdir("/g/data/gb6/BRAN/BRAN2020/month/")
print(contents[0:10])

BRAN2020 w velocities and mixed layer depth
['ocean_u_mth_2008_09.nc', 'atm_flux_diag_mth_2006_06.nc', 'ocean_mld_mth_2022_07.nc', 'ocean_mld_mth_1998_09.nc', 'ocean_u_mth_2000_09.nc', 'ocean_ty_trans_int_z_mth_1993_08.nc', 'ocean_salt_mth_2017_01.nc', 'ocean_mld_mth_2020_11.nc', 'ocean_eta_t_mth_2016_08.nc', 'ocean_v_mth_2001_04.nc']


## Set up the experiments

#### Here, you could add additional parameters to the "info" array
#### These parameters must be named the same as those that are input to the model in the "parameters.py" file in src/
#### If you add another parameter, this must be carried through to the call to main() in the cells below

In [22]:
%%time

expnum = np.arange(1.,17.,1.) + 16
year = np.arange(2000,2016,1)
days = 30
lons = np.array([141.0] * 16)  ### SOTS
lats = np.array([-47.0] * 16)  ### SOTS
atm_co2 = 400.0

lons[lons>180.0] -= 360.0

info = np.array([[expnum[0], year[0], days, lons[0], lats[0], atm_co2],
                 [expnum[1], year[1], days, lons[1], lats[1], atm_co2],
                 [expnum[2], year[2], days, lons[2], lats[2], atm_co2],
                 [expnum[3], year[3], days, lons[3], lats[3], atm_co2],
                 [expnum[4], year[4], days, lons[4], lats[4], atm_co2],
                 [expnum[5], year[5], days, lons[5], lats[5], atm_co2],
                 [expnum[6], year[6], days, lons[6], lats[6], atm_co2],
                 [expnum[7], year[7], days, lons[7], lats[7], atm_co2],
                 [expnum[8], year[8], days, lons[8], lats[8], atm_co2],
                 [expnum[9], year[9], days, lons[9], lats[9], atm_co2],
                 [expnum[10], year[10], days, lons[10], lats[10], atm_co2],
                 [expnum[11], year[11], days, lons[11], lats[11], atm_co2],
                 [expnum[12], year[12], days, lons[12], lats[12], atm_co2],
                 [expnum[13], year[13], days, lons[13], lats[13], atm_co2],
                 [expnum[14], year[14], days, lons[14], lats[14], atm_co2],
                 [expnum[15], year[15], days, lons[15], lats[15], atm_co2]])


CPU times: user 129 μs, sys: 84 μs, total: 213 μs
Wall time: 176 μs


### Make parameter set a pandas DataFrame

In [23]:
%%time

names = ["expnum", "year", "days", "longitude", "latitude", "atmCO2"]
paramsets = pd.DataFrame(info, columns=names)
paramsets


CPU times: user 220 μs, sys: 135 μs, total: 355 μs
Wall time: 365 μs


,expnum,year,days,longitude,latitude,atmCO2
0,17.0,2000.0,30.0,141.0,-47.0,400.0
1,18.0,2001.0,30.0,141.0,-47.0,400.0
2,19.0,2002.0,30.0,141.0,-47.0,400.0
3,20.0,2003.0,30.0,141.0,-47.0,400.0
4,21.0,2004.0,30.0,141.0,-47.0,400.0
5,22.0,2005.0,30.0,141.0,-47.0,400.0
6,23.0,2006.0,30.0,141.0,-47.0,400.0
7,24.0,2007.0,30.0,141.0,-47.0,400.0
8,25.0,2008.0,30.0,141.0,-47.0,400.0
9,26.0,2009.0,30.0,141.0,-47.0,400.0


## Run the experiments

In [24]:
%%time

# setup the log file for each experiment
def setup_logging():
    log_file = f"experiment_output_{multiprocessing.current_process().pid}.log"
    logging.basicConfig(filename=log_file, level=logging.INFO, format="%(asctime)s - %(message)s")


# Define function to run one experiment
def run_experiment(exp):

    setup_logging()
    
    expnum, yr, dayl, lon, lat, atm_co2 = exp
    
    logging.info(f"\n🚀 Running Experiment with Params: {exp}")
    logging.info(f"Process {multiprocessing.current_process().pid} started at {datetime.now()}")
    
    # Run the main function with these parameters
    main(expnum, yr, dayl, lon, lat, atm_co2)

    logging.info(f"Process {multiprocessing.current_process().pid} finished at {datetime.now()}")
    logging.info(f"✅ Experiment Complete: {exp}")
    


CPU times: user 6 μs, sys: 5 μs, total: 11 μs
Wall time: 17.2 μs


### Cut out the experiments already run

This code looks at the output netcdf files to check if some experiments have already completed and will remove these from the list of experiments that are run in the next cell

In [25]:
%%time

done = set()
for fn in os.listdir(OUTPUT_DIR):
    if "exp" in fn and fn.endswith(".nc"):
        print(fn)
        try:
            # Extract the experiment number
            exp_str = fn.split("exp")[-1].split(".nc")[0]
            expnum = int(float(exp_str))
            done.add(expnum)
        except ValueError:
            # In case conversion fails, skip this file
            continue

todo = [exp for exp in info if exp[0] not in done]
np.shape(todo)
todo

lite_year2000.0_30.0days_47S_141E_400.0ppm_exp1.0.nc
lite_year2009.0_30.0days_47S_141E_400.0ppm_exp10.0.nc
lite_year2005.0_30.0days_47S_141E_400.0ppm_exp6.0.nc
lite_year2013.0_30.0days_47S_141E_400.0ppm_exp14.0.nc
lite_year2001.0_30.0days_47S_141E_400.0ppm_exp2.0.nc
lite_year2012.0_30.0days_47S_141E_400.0ppm_exp13.0.nc
lite_year2011.0_30.0days_47S_141E_400.0ppm_exp12.0.nc
lite_year2014.0_30.0days_47S_141E_400.0ppm_exp15.0.nc
lite_year2010.0_30.0days_47S_141E_400.0ppm_exp11.0.nc
lite_year2015.0_30.0days_47S_141E_400.0ppm_exp16.0.nc
lite_year2003.0_30.0days_47S_141E_400.0ppm_exp4.0.nc
lite_year2007.0_30.0days_47S_141E_400.0ppm_exp8.0.nc
lite_year2006.0_30.0days_47S_141E_400.0ppm_exp7.0.nc
lite_year2008.0_30.0days_47S_141E_400.0ppm_exp9.0.nc
lite_year2004.0_30.0days_47S_141E_400.0ppm_exp5.0.nc
lite_year2002.0_30.0days_47S_141E_400.0ppm_exp3.0.nc
CPU times: user 0 ns, sys: 2.7 ms, total: 2.7 ms
Wall time: 2.15 ms


[array([  17., 2000.,   30.,  141.,  -47.,  400.]),
 array([  18., 2001.,   30.,  141.,  -47.,  400.]),
 array([  19., 2002.,   30.,  141.,  -47.,  400.]),
 array([  20., 2003.,   30.,  141.,  -47.,  400.]),
 array([  21., 2004.,   30.,  141.,  -47.,  400.]),
 array([  22., 2005.,   30.,  141.,  -47.,  400.]),
 array([  23., 2006.,   30.,  141.,  -47.,  400.]),
 array([  24., 2007.,   30.,  141.,  -47.,  400.]),
 array([  25., 2008.,   30.,  141.,  -47.,  400.]),
 array([  26., 2009.,   30.,  141.,  -47.,  400.]),
 array([  27., 2010.,   30.,  141.,  -47.,  400.]),
 array([  28., 2011.,   30.,  141.,  -47.,  400.]),
 array([  29., 2012.,   30.,  141.,  -47.,  400.]),
 array([  30., 2013.,   30.,  141.,  -47.,  400.]),
 array([  31., 2014.,   30.,  141.,  -47.,  400.]),
 array([  32., 2015.,   30.,  141.,  -47.,  400.])]

In [26]:
%%time

# Run experiments in parallel using multiprocessing
if __name__ == "__main__":
    num_processes = min(min(multiprocessing.cpu_count(), 16), len(todo))  # Use at most the available cores to a max of 16
    print("Running %i parallel experiments"%(num_processes))
    with multiprocessing.Pool(processes=num_processes) as pool:
        pool.map(run_experiment, todo)

    # Check generated output files
    output_files = os.listdir("output")
    print("\nGenerated output files:", output_files)
    

Running 16 parallel experiments

Generated output files: ['.ipynb_checkpoints']
CPU times: user 58.6 ms, sys: 112 ms, total: 170 ms
Wall time: 5min 19s
